<a href="https://colab.research.google.com/github/jma199/telco-customer-churn/blob/main/Customer_Churn_Dataset_Merge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Telco Customer Churn Data

The Telco customer churn dataset is 5 separate data tables. These data tables need to be merged into one for EDA.

In [1]:
import pandas as pd

# Load files
demographics = pd.read_excel("Telco_customer_churn_demographics.xlsx")
location = pd.read_excel("Telco_customer_churn_location.xlsx")
population = pd.read_excel("Telco_customer_churn_population.xlsx")
services = pd.read_excel("Telco_customer_churn_services.xlsx")
status = pd.read_excel("Telco_customer_churn_status.xlsx")

In [2]:
dfs = {
    "demographics": demographics,
    "location": location,
    "population": population,
    "services": services,
    "status": status
}

for name, df in dfs.items():
    print(f"{name}:", df.shape)
    print(df.columns)
    print("-" * 40)

demographics: (7043, 9)
Index(['Customer ID', 'Count', 'Gender', 'Age', 'Under 30', 'Senior Citizen',
       'Married', 'Dependents', 'Number of Dependents'],
      dtype='object')
----------------------------------------
location: (7043, 9)
Index(['Customer ID', 'Count', 'Country', 'State', 'City', 'Zip Code',
       'Lat Long', 'Latitude', 'Longitude'],
      dtype='object')
----------------------------------------
population: (1671, 3)
Index(['ID', 'Zip Code', 'Population'], dtype='object')
----------------------------------------
services: (7043, 30)
Index(['Customer ID', 'Count', 'Quarter', 'Referred a Friend',
       'Number of Referrals', 'Tenure in Months', 'Offer', 'Phone Service',
       'Avg Monthly Long Distance Charges', 'Multiple Lines',
       'Internet Service', 'Internet Type', 'Avg Monthly GB Download',
       'Online Security', 'Online Backup', 'Device Protection Plan',
       'Premium Tech Support', 'Streaming TV', 'Streaming Movies',
       'Streaming Music', 'Unli

In [3]:
def find_common_columns_in_dfs(dfs_dict, df_keys=None):
    """
    Finds common columns among a specified subset of DataFrames in a dictionary.

    Args:
        dfs_dict (dict): A dictionary where keys are DataFrame names (str) and
                         values are pandas DataFrames.
        df_keys (list, optional): A list of DataFrame names (keys from dfs_dict)
                                   to consider. If None, all DataFrames in dfs_dict
                                   will be considered.

    Returns:
        set: A set of column names common to all specified DataFrames.
             Returns an empty set if no common columns are found or if
             less than two DataFrames are specified.
    """
    if df_keys is None:
        df_keys = list(dfs_dict.keys())

    if not df_keys: # No dataframes specified
        return set()

    # Initialize with columns from the first dataframe
    try:
        common_cols = set(dfs_dict[df_keys[0]].columns)
    except KeyError:
        print(f"Error: Dataframe '{df_keys[0]}' not found in dfs_dict.")
        return set()

    # Intersect with columns from subsequent dataframes
    for key in df_keys[1:]:
        try:
            common_cols = common_cols.intersection(set(dfs_dict[key].columns))
        except KeyError:
            print(f"Warning: Dataframe '{key}' not found in dfs_dict. Skipping this dataframe.")
            continue
    return common_cols

# --- Demonstrating usage of the new function more concisely ---

demo_cases = [
    {
        "description": "across ALL dataframes",
        "keys": list(dfs.keys())
    },
    {
        "description": "among customer-related dataframes",
        "keys": ['demographics', 'location', 'services', 'status']
    },
    {
        "description": "between demographics and location",
        "keys": ['demographics', 'location']
    },
    {
        "description": "between population and location (sharing 'Zip Code')",
        "keys": ['population', 'location']
    }
]

for case in demo_cases:
    case_keys = case["keys"]
    common_cols = find_common_columns_in_dfs(dfs, case_keys)
    print(f"Common columns {case['description']} ({', '.join(case_keys)}): {common_cols}\n")

Common columns across ALL dataframes (demographics, location, population, services, status): set()

Common columns among customer-related dataframes (demographics, location, services, status): {'Customer ID', 'Count'}

Common columns between demographics and location (demographics, location): {'Customer ID', 'Count'}

Common columns between population and location (sharing 'Zip Code') (population, location): {'Zip Code'}



Four of the five tables all include Customer ID, so use it to merge them together. The population data will be skipped.

In [4]:
# Merge location + demographics
customer_df = location.merge(demographics, on="Customer ID", how="left")

In [5]:
# Merge services
customer_df = customer_df.merge(services, on="Customer ID", how="left")

In [6]:
# Merge status
customer_df = customer_df.merge(status.drop(columns='Count'), on="Customer ID", how="left")

In [7]:
# Merge population
# customer_df = customer_df.merge(population, on="Zip Code", how="left")

In [8]:
# Double check that all the customer ID values are unique
customer_df["Customer ID"].nunique(), len(customer_df)

(7043, 7043)

In [9]:
# Verify dataframe
print(f'Dataframe shape is {customer_df.shape}')
print(customer_df.isnull().sum())

Dataframe shape is (7043, 55)
Customer ID                             0
Count_x                                 0
Country                                 0
State                                   0
City                                    0
Zip Code                                0
Lat Long                                0
Latitude                                0
Longitude                               0
Count_y                                 0
Gender                                  0
Age                                     0
Under 30                                0
Senior Citizen                          0
Married                                 0
Dependents                              0
Number of Dependents                    0
Count                                   0
Quarter_x                               0
Referred a Friend                       0
Number of Referrals                     0
Tenure in Months                        0
Offer                                3877
Phon

### Column Clean-Up

Remove any duplicated columns created by the merge.

In [10]:
def remove_y_columns(df):
    columns_to_drop = [col for col in df.columns if '_y' in col]
    if columns_to_drop:
        print(f"Dropping columns: {columns_to_drop}")
        df = df.drop(columns=columns_to_drop)
    else:
        print("No columns with '_y' suffix found.")
    return df

customer_df = remove_y_columns(customer_df)

Dropping columns: ['Count_y', 'Quarter_y']


In [11]:
def rename_x_columns(df):
    new_columns = []
    x_columns = [col for col in df.columns if '_x' in col]
    for col in df.columns:
        if col.endswith('_x'):
            new_columns.append(col[:-2])  # Remove last two characters '_x'
        else:
            new_columns.append(col)
    df.columns = new_columns
    print(f"Renamed columns: {x_columns}")
    return df

customer_df = rename_x_columns(customer_df)

Renamed columns: ['Count_x', 'Quarter_x']


In [12]:
def handle_duplicate_columns(df):
    """
    Identifies and removes duplicate columns from a DataFrame, keeping the first occurrence.
    Prints information about duplicate columns found and the result of the removal.
    """
    duplicate_cols_found = df.columns[df.columns.duplicated()]
    if not duplicate_cols_found.empty:
        print(f"Found duplicate column names: {list(duplicate_cols_found)}")
        # Remove duplicate columns, keeping the first occurrence
        df = df.loc[:, ~df.columns.duplicated()]
        print("Duplicate columns removed (keeping first occurrence).")
    else:
        print("No duplicate column names found.")

    # Verify no duplicates remain
    final_duplicate_check = df.columns[df.columns.duplicated()]
    if not final_duplicate_check.empty:
        print(f"After removal, still found duplicate column names: {list(final_duplicate_check)}")
    else:
        print("All column names are unique after processing.")
    return df

customer_df = handle_duplicate_columns(customer_df)

Found duplicate column names: ['Count']
Duplicate columns removed (keeping first occurrence).
All column names are unique after processing.


### Missing Values

Based on the column name, there are columns that have complementary data.

In [13]:
def fill_missing_values(df, columns):
  '''
  Fill missing values in specified columns with 'None'.

  Parameters:
  - df: DataFrame
  - columns: List of column names to fill missing values in
  '''
  for col in columns:
      if df[col].isnull().any():  # Check if there are any NaN values
          df[col] = df[col].fillna('None')
          print(f"Filled NaN values in '{col}' with 'None'.")
      else:
          print(f"No NaN values found in '{col}'. Skipping fillna.")
      print("-" * 40)

fill_missing_values(customer_df, ["Internet Type", "Internet Service","Churn Category", "Churn Reason", "Offer"])

Filled NaN values in 'Internet Type' with 'None'.
----------------------------------------
No NaN values found in 'Internet Service'. Skipping fillna.
----------------------------------------
Filled NaN values in 'Churn Category' with 'None'.
----------------------------------------
Filled NaN values in 'Churn Reason' with 'None'.
----------------------------------------
Filled NaN values in 'Offer' with 'None'.
----------------------------------------


In [14]:
def compare_internet_service_type_counts(df):
    no_internet_service_count = df[df['Internet Service'] == 'No'].shape[0]
    none_internet_type_count = df[df['Internet Type'] == 'None'].shape[0]

    print(f"Number of 'No' Internet Service: {no_internet_service_count}")
    print(f"Number of 'None' Internet Type: {none_internet_type_count}")

    if no_internet_service_count == none_internet_type_count:
        print("The counts are equal.")
    else:
        print("The counts are not equal.")

compare_internet_service_type_counts(customer_df)

Number of 'No' Internet Service: 1526
Number of 'None' Internet Type: 1526
The counts are equal.


## Export Merged Table

In [15]:
# Merged dataset for EDA
customer_df.to_excel("Telco_customer_churn.xlsx", index=False)